In [ ]:
# ------------------------------------------------------------
# Cell 1 — Environment setup
# ------------------------------------------------------------
# 1. Check the GPU (T4 is fine).
# 2. Pin numpy < 2.0 so compiled deps behave.
# 3. Install the core stack (torch+transformers+faiss+bitsandbytes+peft).
#    NOTE: we do NOT install `datasets` or `trl` anymore.
# ------------------------------------------------------------

!nvidia-smi

# Pin numpy to a 1.x version to avoid binary issues.
!pip install -q "numpy<2"

# Install the rest of the stack. Let pip resolve compatible versions.
!pip install -q \
  "transformers[torch]" \
  accelerate \
  bitsandbytes \
  peft \
  sentence-transformers \
  faiss-cpu \
  tqdm \
  huggingface_hub

import numpy as np, torch, transformers
print("NumPy version:", np.__version__)
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

In [ ]:
# ------------------------------------------------------------
# Cell 2 — Patch torch if needed, then import libraries
# ------------------------------------------------------------
# Some transformer internals expect torch.library.register_fake to exist.
# On some Colab builds it doesn't, causing:
#   AttributeError: module 'torch.library' has no attribute 'register_fake'
#
# We define a harmless no-op register_fake if it's missing, then
# import everything else we need.
# ------------------------------------------------------------

import torch
import types

print("Torch version:", torch.__version__)

# Ensure torch.library exists.
if not hasattr(torch, "library"):
    torch.library = types.SimpleNamespace()

# Provide a dummy register_fake if missing.
if not hasattr(torch.library, "register_fake"):
    def _dummy_register_fake(*args, **kwargs):
        def decorator(fn):
            return fn
        return decorator
    torch.library.register_fake = _dummy_register_fake
    print("Patched torch.library.register_fake with a no-op.")
else:
    print("torch.library.register_fake already present.")

# Now import the rest.
import os
import json
from typing import List, Dict, Tuple

import numpy as np
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from torch.utils.data import Dataset as TorchDataset

print("All imports successful.")

In [ ]:
# ------------------------------------------------------------
# Cell 3 — Upload the RAG-ready JSONL file
# ------------------------------------------------------------
# Colab will show a file picker. Upload the file you prepared
# (e.g., rag_ready_docs_20251111.jsonl). We detect whichever .jsonl
# you upload and use that.
# ------------------------------------------------------------

from google.colab import files

uploaded = files.upload()
JSONL_PATH = None

for fname in uploaded.keys():
    print("Uploaded:", fname)
    if fname.endswith(".jsonl"):
        JSONL_PATH = fname

if JSONL_PATH is None:
    raise ValueError("Please upload your RAG-ready .jsonl file.")

print("Using JSONL file:", JSONL_PATH)

In [ ]:
# ------------------------------------------------------------
# Cell 4 — Load and clean corpus from JSONL
# ------------------------------------------------------------
# Your JSONL text often has patterns like:
#   page_content='...text...' metadata={...}
#
# We:
#  - Strip "page_content='...'"
#  - Remove trailing " metadata={'...'}" if embedded in text
#  - Normalize whitespace
#  - Keep: id, cleaned text, metadata
# ------------------------------------------------------------

def clean_text(raw_text: str) -> str:
    text = raw_text

    # Remove leading "page_content='"
    if text.startswith("page_content='"):
        text = text[len("page_content='"):]
        if text.endswith("'"):
            text = text[:-1]

    # Remove trailing metadata blob if present
    if " metadata={'" in text:
        text = text.split(" metadata={'", 1)[0]

    # Clean whitespace
    text = text.replace("\\n", " ").replace("\n", " ")
    text = " ".join(text.split())

    return text


def load_corpus(jsonl_path: str):
    docs = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)

            text = clean_text(obj.get("text", ""))
            if not text:
                continue

            docs.append({
                "id": obj.get("id", ""),
                "text": text,
                "metadata": obj.get("metadata", {}),
            })

    print(f"Loaded {len(docs)} chunks from dataset.")
    return docs


corpus = load_corpus(JSONL_PATH)
print("Example chunk:\n", corpus[0])

In [ ]:
# ------------------------------------------------------------
# Cell 5 — Build embeddings + FAISS HNSW index
# ------------------------------------------------------------
# We:
#   - Embed each chunk using BAAI/bge-large-en-v1.5
#   - Normalize embeddings (for cosine-like similarity)
#   - Build a FAISS HNSW index
#   - Save index and corpus metadata for reuse
# ------------------------------------------------------------

EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"
INDEX_PATH = "faiss_index.bin"
CORPUS_PATH = "corpus_meta.json"

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


def build_embeddings(docs: List[Dict], model_name: str) -> np.ndarray:
    model = SentenceTransformer(model_name)
    texts = [d["text"] for d in docs]
    prefixed = [QUERY_PREFIX + t for t in texts]

    embeddings = model.encode(
        prefixed,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    print("Embeddings shape:", embeddings.shape)
    return embeddings


def build_faiss_hnsw(embeddings: np.ndarray) -> faiss.Index:
    dim = embeddings.shape[1]
    index = faiss.IndexHNSWFlat(dim, 32)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = 64

    index.add(embeddings)
    print("FAISS index size:", index.ntotal)
    return index


embeddings = build_embeddings(corpus, EMBED_MODEL_NAME)
index = build_faiss_hnsw(embeddings)

faiss.write_index(index, INDEX_PATH)
with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print("Saved index to:", INDEX_PATH)
print("Saved corpus to:", CORPUS_PATH)

In [ ]:
# ------------------------------------------------------------
# Cell 6 — Define RAGEngine (Qwen2.5-7B-Instruct)
# ------------------------------------------------------------
# RAGEngine:
#   - Uses BGE for query embeddings
#   - Uses FAISS HNSW for retrieval
#   - Uses Qwen/Qwen2.5-7B-Instruct as the LLM (4-bit)
#   - Can optionally attach a LoRA adapter (for the fine-tuned model)
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TOP_K = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def build_chatml_prompt(system_prompt: str, user_prompt: str) -> str:
    """
    Build a ChatML-style prompt for Qwen:
      <|im_start|>system
      ...
      <|im_end|>
      <|im_start|>user
      ...
      <|im_end|>
      <|im_start|>assistant
      ... (model continues)
    """
    return (
        "<|im_start|>system\n" + system_prompt + "<|im_end|>\n"
        "<|im_start|>user\n" + user_prompt + "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


class RAGEngine:
    def __init__(self,
                 index_path: str,
                 corpus_path: str,
                 embed_model_name: str,
                 model_name: str,
                 use_lora_adapter: str = None):
        # Load index and corpus.
        print("Loading FAISS index and corpus...")
        self.index = faiss.read_index(index_path)
        with open(corpus_path, "r", encoding="utf-8") as f:
            self.corpus = json.load(f)

        # Embedding model.
        print("Loading embedding model:", embed_model_name)
        self.embed_model = SentenceTransformer(embed_model_name)

        # Tokenizer.
        print("Loading tokenizer:", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            use_fast=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # 4-bit quantization config for T4.
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

        print("Loading Qwen model in 4-bit...")
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
        )

        # Attach LoRA adapter if provided.
        if use_lora_adapter is not None:
            print("Attaching LoRA adapter from:", use_lora_adapter)
            self.model = PeftModel.from_pretrained(
                base_model,
                use_lora_adapter,
                torch_dtype=torch.bfloat16,
                device_map="auto",
            )
        else:
            self.model = base_model

        self.model.eval()

    def embed_query(self, query: str) -> np.ndarray:
        text = QUERY_PREFIX + query
        emb = self.embed_model.encode(
            [text],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return emb

    def retrieve(self, query: str, k: int = TOP_K) -> List[Dict]:
        q_emb = self.embed_query(query)
        D, I = self.index.search(q_emb, k)

        results = []
        for idx, score in zip(I[0], D[0]):
            doc = self.corpus[idx]
            results.append({
                "id": doc["id"],
                "text": doc["text"],
                "metadata": doc.get("metadata", {}),
                "score": float(score),
            })
        return results

    def build_prompt(self, query: str, contexts: List[Dict]) -> str:
        context_lines = []
        for i, c in enumerate(contexts, start=1):
            meta_str = ", ".join(f"{k}={v}" for k, v in c.get("metadata", {}).items())
            context_lines.append(
                f"[{i}] (id={c['id']}, {meta_str})\n{c['text']}\n"
            )
        context_block = "\n\n".join(context_lines)

        system_instruction = (
            "You are an MScAC interview mentor at the University of Toronto. "
            "Use ONLY the provided context to answer. If something is not in the context, "
            "say you are not sure and suggest asking the MScAC office or the official website.\n"
            "Your response MUST follow this structure:\n"
            "1. Direct Answer\n"
            "2. How to Prepare (numbered steps)\n"
            "3. Sources (bullet list of [chunk-id] you used)\n"
        )

        user_message = (
            f"Question: {query}\n\n"
            f"Context:\n{context_block}\n"
        )

        prompt = build_chatml_prompt(system_instruction, user_message)
        return prompt

    @torch.no_grad()
    def answer(self, query: str, max_new_tokens: int = 512) -> Tuple[str, List[Dict]]:
        contexts = self.retrieve(query, TOP_K)
        prompt = self.build_prompt(query, contexts)

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096,
        ).to(DEVICE)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        generated = outputs[0, inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(generated, skip_special_tokens=True)
        return text, contexts


In [ ]:
# ------------------------------------------------------------
# Cell 7 — Baseline: Qwen2.5-7B + RAG
# ------------------------------------------------------------
# We:
#   - Instantiate a RAGEngine with base Qwen (no fine-tuning)
#   - Ask 5 MScAC interview-prep questions
#   - Store the answers for later comparison
# ------------------------------------------------------------

print("Creating RAG engine with base Qwen model...")
base_rag = RAGEngine(
    index_path=INDEX_PATH,
    corpus_path=CORPUS_PATH,
    embed_model_name=EMBED_MODEL_NAME,
    model_name=MODEL_NAME,
    use_lora_adapter=None,
)

EVAL_QUESTIONS = [
    "How should I prepare for MScAC technical interviews?",
    "What are the key things to highlight from my MScAC coursework during interviews?",
    "How can I explain the MScAC program structure to an interviewer?",
    "How should I talk about my AI projects in MScAC interviews?",
    "What behavioral questions should I expect for MScAC internships and how do I prepare?",
]

base_results = {}

for q in EVAL_QUESTIONS:
    print(f"\n[BASE] Answering: {q}")
    ans, _ = base_rag.answer(q, max_new_tokens=384)
    base_results[q] = ans

sample_q = EVAL_QUESTIONS[0]
print("\n=== SAMPLE BASE ANSWER ===")
print("Question:", sample_q)
print("\nAnswer:\n", base_results[sample_q])

In [ ]:
# ------------------------------------------------------------
# Cell 8 — Build interview-style SFT records from corpus
# ------------------------------------------------------------
# For each chunk (up to MAX_CHUNKS_FOR_TRAIN), we create:
#   - system: MScAC mentor description
#   - user: "Using the following MScAC resource, explain what a student should know..."
#   - assistant: 3-part answer (Direct Answer / How to Prepare / Sources)
# We'll later convert these into ChatML strings for training.
# ------------------------------------------------------------

MAX_CHUNKS_FOR_TRAIN = 300   # balanced for T4
MAX_CHARS_PER_CHUNK = 800    # limit chunk length per example


def build_interview_style_sft_records_from_corpus(
    docs,
    max_chunks: int = MAX_CHUNKS_FOR_TRAIN,
    max_chars: int = MAX_CHARS_PER_CHUNK,
):
    records = []

    for i, doc in enumerate(docs[:max_chunks]):
        text = doc["text"][:max_chars]
        chunk_id = doc["id"]

        system_prompt = (
            "You are an MScAC interview mentor at the University of Toronto. "
            "You explain information clearly, connect it to MScAC interview preparation, "
            "and keep your answers structured and concise."
        )

        user_prompt = (
            "Using the following MScAC resource, explain what a student should know "
            "for MScAC interviews and how they should prepare. "
            "Focus on key ideas and practical interview preparation advice.\n\n"
            f"{text}"
        )

        assistant_answer = f"""1. Direct Answer
{text}

2. How to Prepare (for MScAC interviews)
- Read this content carefully and highlight 2–3 key points that could come up in an interview.
- Practice explaining these points clearly in your own words, as if you were talking to an MScAC interviewer.
- Relate each key point to your own projects, experiences, or goals so you can give concrete examples.
- Anticipate follow-up questions about these details (e.g., why they matter, how they influence your decisions).

3. Sources
- Derived from MScAC resource chunk id={chunk_id}"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_answer},
        ]
        records.append({"messages": messages})

    print(f"Built {len(records)} interview-style SFT records from corpus.")
    return records


sft_records = build_interview_style_sft_records_from_corpus(corpus)
print("Example SFT record:\n", sft_records[0])


In [ ]:
# ------------------------------------------------------------
# Cell 9 — Convert SFT records into ChatML text strings
# ------------------------------------------------------------
# We'll get a list of raw text sequences ("train_texts")
# that we can feed into a simple PyTorch Dataset.
# ------------------------------------------------------------

def build_chatml_from_messages(messages: List[Dict]) -> str:
    parts = []
    for m in messages:
        role = m["role"]
        content = m["content"]
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    return "\n".join(parts)


train_texts = [build_chatml_from_messages(r["messages"]) for r in sft_records]

print("Number of training examples:", len(train_texts))
print("Sample training text snippet:\n")
print(train_texts[0][:700])

In [ ]:
# ------------------------------------------------------------
# Cell 10 — Define TextCausalDataset for QLoRA training
# ------------------------------------------------------------
# We build a simple torch.utils.data.Dataset that:
#   - Takes a list of ChatML strings
#   - Tokenizes each to fixed length
#   - Returns input_ids, attention_mask, labels (same as input_ids)
# This avoids using `datasets` and keeps things simple and robust.
# ------------------------------------------------------------

class TextCausalDataset(TorchDataset):
    def __init__(self, texts, tokenizer, max_length=896):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        labels = input_ids.clone()
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

print("TextCausalDataset class defined.")

In [ ]:
# ------------------------------------------------------------
# Cell 11 — QLoRA fine-tuning on Qwen2.5-7B-Instruct (forced GPU)
# ------------------------------------------------------------
# Fixes:
#   - Force the whole 4-bit model onto GPU with device_map={"": 0}
#   - Use prepare_model_for_kbit_training for proper 4-bit training
#   - Turn off use_cache for training
# ------------------------------------------------------------

OUTPUT_DIR = "qwen_mscac_qlora"

bnb_config_train = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("Loading tokenizer for Qwen training...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Building training dataset...")
train_dataset = TextCausalDataset(train_texts, tokenizer, max_length=896)

print("Loading base Qwen in 4-bit for QLoRA training (forced GPU)...")
base_model_train = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config_train,
    device_map={"": 0},        # <--- force everything on GPU 0
    low_cpu_mem_usage=True,
)

# Prepare model for 4-bit training (sets up gradients / norm layers, etc.)
print("Preparing model for 4-bit training...")
base_model_train = prepare_model_for_kbit_training(
    base_model_train,
    use_gradient_checkpointing=True,
)

# Disable cache for training (needed with checkpointing)
base_model_train.config.use_cache = False

# Wrap with LoRA
print("Applying LoRA config...")
model_train = get_peft_model(base_model_train, lora_config)
model_train.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                 # can increase later if it runs fine
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),     # fp16 is fine on T4
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,       # we already enabled it via prepare_model_for_kbit_training
    report_to="none",
    max_grad_norm=0.3,
    remove_unused_columns=False,
)

print("Starting training with Trainer...")
trainer = Trainer(
    model=model_train,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

print("Saving LoRA adapter and tokenizer...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("QLoRA training finished. Adapter saved in:", OUTPUT_DIR)

In [ ]:
# ------------------------------------------------------------
# Cell 12 — Free training model from GPU memory
# ------------------------------------------------------------
# After training, free the training model and clear CUDA cache.
# This makes room for the inference-time RAG engine.
# ------------------------------------------------------------

del trainer
del model_train
del base_model_train
torch.cuda.empty_cache()

print("Cleared training models from GPU memory.")

In [ ]:
# ------------------------------------------------------------
# Cell 13 — RAG with fine-tuned Qwen (LoRA adapter)
# ------------------------------------------------------------
# We:
#   - Create a new RAGEngine
#   - Attach the LoRA adapter we just trained
#   - Answer the same evaluation questions
#   - Store fine-tuned answers in ft_results
# ------------------------------------------------------------

print("Creating RAG engine with fine-tuned Qwen (LoRA adapter)...")
finetuned_rag = RAGEngine(
    index_path=INDEX_PATH,
    corpus_path=CORPUS_PATH,
    embed_model_name=EMBED_MODEL_NAME,
    model_name=MODEL_NAME,
    use_lora_adapter=OUTPUT_DIR,
)

ft_results = {}

for q in EVAL_QUESTIONS:
    print(f"\n[FINE-TUNED] Answering: {q}")
    ans, _ = finetuned_rag.answer(q, max_new_tokens=384)
    ft_results[q] = ans

print("\nCollected fine-tuned answers for all evaluation questions.")

In [ ]:
# ------------------------------------------------------------
# Cell 14 — Compare base vs fine-tuned + optional chat + summary
# ------------------------------------------------------------
# 1. Show base vs fine-tuned answers for each question (for your team).
# 2. Optional interactive chat loop with the fine-tuned RAG bot.
# 3. A short summary of the pipeline for your slides.
# ------------------------------------------------------------

def show_comparison(questions, base_dict, ft_dict):
    for i, q in enumerate(questions, start=1):
        print("=" * 120)
        print(f"[{i}] QUESTION:\n{q}\n")

        print("---- BASE Qwen + RAG ----")
        print(base_dict[q][:2000])
        print("\n---- FINE-TUNED Qwen + RAG ----")
        print(ft_dict[q][:2000])
        print("\n")

show_comparison(EVAL_QUESTIONS, base_results, ft_results)


def chat_loop(rag_engine: RAGEngine):
    print("MScAC Interview Mentor Chatbot (type 'exit' to quit)")
    while True:
        q = input("You: ").strip()
        if not q:
            continue
        if q.lower() in {"exit", "quit"}:
            break
        ans, ctxs = rag_engine.answer(q, max_new_tokens=384)
        print("\n--- Answer ---\n")
        print(ans)
        print("\n--- Context IDs ---")
        for c in ctxs:
            print(f"- {c['id']} (score={c['score']:.4f})")
        print("\n")

# Uncomment this to try the chatbot live:
# chat_loop(finetuned_rag)

print("=== PIPELINE SUMMARY (for slides) ===")
print(f"- Number of RAG chunks indexed: {len(corpus)}")
print(f"- Embedding model: {EMBED_MODEL_NAME}")
print(f"- Index: FAISS HNSW (M=32, efConstruction=200, efSearch=64)")
print(f"- Base LLM: {MODEL_NAME}")
print(f"- QLoRA training samples (ChatML strings): {len(train_texts)}")
print(f"- QLoRA epochs: 1")
print(f"- Max sequence length: 896")
print(f"- LoRA adapter directory: {OUTPUT_DIR}")